# Client Participation: FedGRA vs. High-Loss

Scatter plot showing which clients (ID 0–9) are selected at each communication round.
- 🔵 **High-Loss** (blue circles)
- ❌ **FedGRA** (red × markers)

In [ ]:
import sys
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Add root for style
root_dir = Path("../../..").resolve()
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
from style import MatplotlibStyle
MatplotlibStyle().apply()

print("✅ Imports ready.")


In [ ]:
# ============================================================
# Load & Parse Selection Logs
# ============================================================

FILES = {
    "FedGRA":    "fedgra_keras_exact_mnist_-train-20260615_023523-c2a79bf0_selection_log.csv",
    "High-Loss": "fedgra_keras_exact_high_loss_mnist_-train-20260615_042615-c17bfe7d_selection_log.csv",
}

def parse_client_ids(selected_text: str):
    """
    Parse 'client.3@group-1;client.5@group-1' → [2, 4]  (0-indexed client IDs)
    """
    if pd.isna(selected_text) or not str(selected_text).strip():
        return []
    ids = []
    for token in str(selected_text).split(";"):
        m = re.search(r"client\.(\d+)@", token)
        if m:
            ids.append(int(m.group(1)) - 1)  # client 1–10 → ID 0–9
    return ids

# Build dict: {method_name: {"rounds": [...], "clients": [...]}}
participation = {}

for label, filename in FILES.items():
    filepath = Path(filename)
    if not filepath.exists():
        print(f"⚠️  Missing: {filename}")
        continue
    
    df = pd.read_csv(filepath)
    rounds_list = []
    clients_list = []
    
    for _, row in df.iterrows():
        r = int(row["round"])
        cids = parse_client_ids(row["selected_clients"])
        for cid in cids:
            rounds_list.append(r)
            clients_list.append(cid)
    
    participation[label] = {
        "rounds": np.array(rounds_list),
        "clients": np.array(clients_list),
    }
    unique_clients = np.unique(clients_list)
    print(f"✅ {label:12s}: {len(rounds_list)} selections, "
          f"unique clients = {unique_clients}, rounds = {df['round'].min()}–{df['round'].max()}")


In [ ]:
# ============================================================
# Client Participation Scatter Plot
# ============================================================

fig, ax = plt.subplots(figsize=(16, 5))

# High-Loss: blue circles (plotted first so FedGRA red X's appear on top)
hl = participation["High-Loss"]
ax.scatter(hl["rounds"], hl["clients"],
           marker='o', s=50, c='#2a7de1', edgecolors='white', linewidths=0.4,
           label="High-Loss", zorder=2)

# FedGRA: red X markers
fg = participation["FedGRA"]
ax.scatter(fg["rounds"], fg["clients"],
           marker='x', s=50, c='#e53e3e', linewidths=1.2,
           label="FedGRA", zorder=3)

# Axes
ax.set_xlabel("Communication Round", fontsize=24)
ax.set_ylabel("Client ID", fontsize=24)
ax.set_xlim(-1, 101)
ax.set_ylim(-0.8, 9.8)
ax.set_yticks(range(10))
ax.set_yticklabels([str(i) for i in range(10)])

# Legend
ax.legend(fontsize=18, loc="upper right", frameon=True, fancybox=True,
          markerscale=1.5)

# Title
ax.set_title("Client Participation Result of Simulation Experiment",
             fontsize=22, pad=12)

plt.tight_layout()
plt.show()
